# 3. Reinforcement Learning with Verifiable Rewards (GRPO)
**HLT 2026 - Reasoning with Verifiable Rewards**

This notebook fine-tunes **Qwen2.5-0.5B-Instruct** using **Group Relative Policy Optimization (GRPO)** as introduced in **DeepSeekMath** and **DeepSeek-R1-Zero**:
1. **Zero Critic Architecture**: Eliminates PPO's auxiliary Value Model by generating $G=4$ completions per prompt and calculating relative group advantages ($A_i = \frac{r_i - \mu}{\sigma}$).
2. **Deterministic Verifiable Rewards**: 3 rule-based signals (Accuracy, Format, Validity).
3. **PEFT (LoRA)**: Parameter-efficient fine-tuning that fits comfortably inside free T4 GPU VRAM (15–16 GB) without OOM.

> ⚡ **Quick Test Mode Available**: You can set `DRY_RUN = True` below to run a **fast 1-minute sanity check** on just 4 prompts before launching the full training!

## 1. Install Dependencies (Colab / Kaggle)

In [ ]:
!pip install -q --upgrade "trl>=0.14.0" "transformers>=4.48.0" peft accelerate datasets bitsandbytes "torchao>=0.16.0"


## 2. Hardware Verification & Model Configuration

In [ ]:
# Bypass Colab pre-installed torchao version incompatibility (we use standard LoRA)
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao as lora_torchao
    lora_torchao.is_torchao_available = lambda: False
except (ImportError, AttributeError):
    pass

import os
import ast
import re
import json
import torch
import urllib.request
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig

assert torch.cuda.is_available(), "GPU not detected! Enable GPU via Runtime -> Change runtime type -> T4 GPU"
print(f"Active GPU: {torch.cuda.get_device_name(0)}")

# Model choice: Qwen2.5-1.5B-Instruct (as in project proposal) or Qwen2.5-0.5B-Instruct
# Both fit comfortably inside Colab T4 GPU VRAM (15 GB) with LoRA in FP16!
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Selected model: {MODEL_ID}")


## 3. Load Training Dataset (With Quick Test Toggle)

In [ ]:
# ==============================================================================
# TRAINING MODE SELECTION:
# "fast":    ~5 minutes (25 steps on 100 curriculum-ordered puzzles) -> Real >0% accuracy!
# "full":    ~20 minutes (125 steps on all 1,000 puzzles) -> Maximum accuracy (~50-70%)
# "dry_run": ~20 seconds (2 steps on 4 puzzles) -> Rapid sanity check
# ==============================================================================
TRAINING_MODE = "fast"

data_dir = "data" if not os.path.basename(os.getcwd()) == "notebooks" else "../data"
os.makedirs(data_dir, exist_ok=True)
train_file = os.path.join(data_dir, "train_countdown.jsonl")

if not os.path.exists(train_file):
    print(f"'{train_file}' not found locally. Downloading 1,000 training puzzles from GitHub...")
    url = "https://raw.githubusercontent.com/Gheb6/countdown-grpo-reasoning/main/data/train_countdown.jsonl"
    urllib.request.urlretrieve(url, train_file)

with open(train_file, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

if TRAINING_MODE == "dry_run":
    data = data[:4]
    max_steps = 2
    eval_size = 5
    print(f"⚡ DRY RUN MODE: 2 steps on 4 prompts (~20 seconds).")
elif TRAINING_MODE == "fast":
    # Curriculum Learning: sort by difficulty (2-number and 3-number solvable puzzles first)
    data = sorted(data, key=lambda d: len(re.findall(r'\d+', d.get('example_solution', '0 0 0 0'))))[:100]
    max_steps = 25
    eval_size = 20
    print(f"⚡ FAST TRAINING MODE (Curriculum): 25 steps on 100 easiest puzzles (~5 minutes).")
else:
    max_steps = -1
    eval_size = 50
    print(f"🚀 FULL TRAINING MODE: 1 full epoch on all {len(data)} prompts (~20 minutes).")

SYSTEM_PROMPT = "A conversation between User and Assistant. The user asks a math question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively."

train_prompts = []
for d in data:
    user_content = (
        f"Using the numbers {d['numbers']}, create an arithmetic equation that equals exactly {d['target']}.\n"
        f"Rules:\n"
        f"- You can use basic arithmetic operations (+, -, *, /).\n"
        f"- You do NOT need to use all numbers (you can use 2, 3, or all 4).\n"
        f"- In <think>...</think>, explore combinations and test if they equal the target.\n"
        f"- In <answer>...</answer>, output ONLY the winning equation (e.g. <answer>(60 + 11)</answer>).\n"
    )
    train_prompts.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        "numbers": d["numbers"],
        "target": d["target"]
    })

train_dataset = Dataset.from_list(train_prompts)
print(f"Dataset ready: {len(train_dataset)} instances prepared for RL training.")


## 4. Define Verifiable Reward Functions (R1-Zero Paradigm)

In [ ]:
ALLOWED_OPERATORS = {
    ast.Add: lambda a, b: a + b,
    ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b,
    ast.Div: lambda a, b: a / b if b != 0 else None,
    ast.FloorDiv: lambda a, b: a // b if b != 0 else None,
    ast.USub: lambda a: -a,
    ast.UAdd: lambda a: +a
}

ALLOWED_NODE_TYPES = (
    ast.Expression,
    ast.BinOp,
    ast.UnaryOp,
    ast.Constant,
    ast.Add,
    ast.Sub,
    ast.Mult,
    ast.Div,
    ast.FloorDiv,
    ast.USub,
    ast.UAdd
)

def is_safe_math_ast(tree):
    for node in ast.walk(tree):
        if not isinstance(node, ALLOWED_NODE_TYPES):
            return False
        if isinstance(node, ast.BinOp) and type(node.op) not in ALLOWED_OPERATORS:
            return False
        if isinstance(node, ast.UnaryOp) and type(node.op) not in ALLOWED_OPERATORS:
            return False
        if isinstance(node, ast.Constant):
            if isinstance(node.value, bool) or not isinstance(node.value, (int, float)):
                return False
    return True

def extract_clean_equation(content):
    """Extract raw arithmetic expression from <answer> tags, tolerating harmless prefixes."""
    ans_match = re.search(r"<answer>(.*?)</answer>", content, re.DOTALL | re.IGNORECASE)
    if not ans_match:
        return None
    raw = ans_match.group(1).split("=")[0].strip()
    math_match = re.search(r"[\(\d].*", raw)
    return math_match.group(0).strip() if math_match else raw

def countdown_strict_reward(completions, numbers, target, **kwargs):
    """
    Strict, verifiable reward function following DeepSeek-R1-Zero / TinyZero paradigm.
    CRITICAL ANTI-HACKING RULE: Incorrect mathematical answers receive exactly 0.0 reward!
    Correct answers receive 1.0 accuracy reward + 0.1 format bonus.
    """
    rewards = []
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    for comp, nums, tgt in zip(completions, numbers, target):
        content = comp[0]["content"] if isinstance(comp, list) else comp
        has_format = bool(re.search(pattern, content.strip(), re.DOTALL | re.IGNORECASE))
        
        expr = extract_clean_equation(content)
        if not expr:
            rewards.append(0.0)
            continue
            
        try:
            tree = ast.parse(expr, mode='eval')
            if not is_safe_math_ast(tree):
                rewards.append(0.0)
                continue
                
            used = []
            for n in ast.walk(tree):
                if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
                    used.append(int(n.value) if float(n.value).is_integer() else n.value)
            avail = list(nums)
            valid_nums = True
            for u in used:
                if u in avail:
                    avail.remove(u)
                else:
                    valid_nums = False
                    break
            if not valid_nums or len(used) < 2:
                rewards.append(0.0)
                continue

            def ev(node):
                if isinstance(node, ast.Expression): return ev(node.body)
                elif isinstance(node, ast.Constant): return float(node.value)
                elif isinstance(node, ast.UnaryOp):
                    op = ALLOWED_OPERATORS.get(type(node.op))
                    v = ev(node.operand)
                    return op(v) if op and v is not None else None
                elif isinstance(node, ast.BinOp):
                    op = ALLOWED_OPERATORS.get(type(node.op))
                    l, r = ev(node.left), ev(node.right)
                    return op(l, r) if op and l is not None and r is not None else None
                return None

            computed = ev(tree)
            if computed is not None and abs(computed - tgt) < 1e-6:
                # Correct answer reward: 1.0 + 0.1 format bonus
                score = 1.0 + (0.1 if has_format else 0.0)
                rewards.append(score)
            else:
                # Incorrect math receives strictly 0.0 (prevents reward hacking)
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards


## 5. Configure LoRA (PEFT) & GRPOTrainer

In [ ]:
# Bypass Colab pre-installed torchao version incompatibility (we use standard LoRA)
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao as lora_torchao
    lora_torchao.is_torchao_available = lambda: False
except (ImportError, AttributeError):
    pass

# Bypass T4 GPU compute capability 7.5 bfloat16 addmm limitation in TRL chunked logprob kernel
try:
    import trl.kernels.chunked_logprob as clp
    clp._MM_OUT_DTYPE = False
    clp._addmm_fp32 = lambda acc, a, b: acc.addmm_(a.float(), b.float())
except Exception:
    pass

# Target all linear modules (both attention and MLP computation layers for arithmetic reasoning)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

models_dir = "models" if not os.path.basename(os.getcwd()) == "notebooks" else "../models"
os.makedirs(models_dir, exist_ok=True)

training_args = GRPOConfig(
    output_dir=os.path.join(models_dir, "qwen-countdown-grpo"),
    learning_rate=1e-4, # Optimal learning rate for LoRA adapters in RL
    temperature=0.9,    # Exploration temperature for rollouts
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2 if TRAINING_MODE == "dry_run" else 4,
    num_generations=4, # Group size G=4 for relative reward whitening
    max_completion_length=512,
    num_train_epochs=1,
    max_steps=max_steps,
    logging_steps=1 if TRAINING_MODE == "dry_run" else 5,
    save_steps=50,
    bf16=False,
    fp16=True,
    model_init_kwargs={"torch_dtype": torch.float16},
    report_to="none"
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    reward_funcs=[countdown_strict_reward],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config
)


## 6. Launch GRPO Training

In [ ]:
# Ensure chunked_logprob fallback is active on GPU
try:
    import trl.kernels.chunked_logprob as clp
    clp._MM_OUT_DTYPE = False
    clp._addmm_fp32 = lambda acc, a, b: acc.addmm_(a.float(), b.float())
except Exception:
    pass

print(f"🚀 Launching GRPO RL training ({TRAINING_MODE.upper()} mode)...")
trainer.train()

final_save_path = os.path.join(models_dir, "qwen-countdown-grpo-final")
trainer.model.save_pretrained(final_save_path)
print(f"✅ Training completed! LoRA adapter weights saved to {final_save_path}")


## 7. Evaluate Fine-Tuned GRPO Policy & Final 3-Way Comparative Summary


In [ ]:
import os
import re
import ast
import json
import urllib.request
import torch
from transformers import AutoTokenizer

# 1. Load test dataset
test_file = os.path.join(data_dir, "test_countdown.jsonl")
if not os.path.exists(test_file):
    print("Downloading test puzzles from GitHub...")
    url = "https://raw.githubusercontent.com/Gheb6/countdown-grpo-reasoning/main/data/test_countdown.jsonl"
    urllib.request.urlretrieve(url, test_file)

with open(test_file, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

eval_samples = test_data[:eval_size]
print(f"🧪 Evaluating fine-tuned GRPO model on {len(eval_samples)} unseen test puzzles...\n")

# 2. Setup tokenizer and evaluation mode
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
trainer.model.eval()
device = next(trainer.model.parameters()).device

correct_count = 0
total_tokens = 0

for i, item in enumerate(eval_samples):
    nums = item["numbers"]
    tgt = item["target"]
    
    user_content = (
        f"Using the numbers {nums}, create an arithmetic equation that equals exactly {tgt}.\n"
        f"Rules:\n"
        f"- You can use basic arithmetic operations (+, -, *, /).\n"
        f"- You do NOT need to use all numbers (you can use 2, 3, or all 4).\n"
        f"- In <think>...</think>, explore combinations and test if they equal the target.\n"
        f"- In <answer>...</answer>, output ONLY the winning equation (e.g. <answer>(60 + 11)</answer>).\n"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out_ids = trainer.model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    gen_tokens = out_ids[0][inputs.input_ids.shape[1]:]
    completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)
    total_tokens += len(gen_tokens)
    
    # Check correctness via AST
    is_correct = False
    expr = extract_clean_equation(completion)
    if expr:
        try:
            tree = ast.parse(expr, mode='eval')
            if is_safe_math_ast(tree):
                used = []
                for n in ast.walk(tree):
                    if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
                        used.append(int(n.value) if float(n.value).is_integer() else n.value)
                avail = list(nums)
                valid = True
                for u in used:
                    if u in avail: avail.remove(u)
                    else: valid = False; break
                if valid and len(used) >= 2:
                    def ev(node):
                        if isinstance(node, ast.Expression): return ev(node.body)
                        elif isinstance(node, ast.Constant): return float(node.value)
                        elif isinstance(node, ast.UnaryOp):
                            op = ALLOWED_OPERATORS.get(type(node.op))
                            v = ev(node.operand)
                            return op(v) if op and v is not None else None
                        elif isinstance(node, ast.BinOp):
                            op = ALLOWED_OPERATORS.get(type(node.op))
                            l, r = ev(node.left), ev(node.right)
                            return op(l, r) if op and l is not None and r is not None else None
                        return None
                    computed = ev(tree)
                    if computed is not None and abs(computed - tgt) < 1e-6:
                        is_correct = True
        except Exception:
            pass
            
    if is_correct:
        correct_count += 1
    
    status_icon = "✅" if is_correct else "❌"
    print(f"[{i+1}/{len(eval_samples)}] Target: {tgt} | Numbers: {nums} -> {status_icon}")
    if i < 3:
        preview = completion.replace('\n', ' ')[:140]
        print(f"    ↳ Completion: {preview}...")

grpo_accuracy = (correct_count / len(eval_samples)) * 100
avg_tokens = total_tokens / len(eval_samples)

# 3. Load baseline results
results_dir = "results" if not os.path.basename(os.getcwd()) == "notebooks" else "../results"
os.makedirs(results_dir, exist_ok=True)
base_path = os.path.join(results_dir, "baselines.json")
if not os.path.exists(base_path):
    try:
        url = "https://raw.githubusercontent.com/Gheb6/countdown-grpo-reasoning/main/results/baselines.json"
        urllib.request.urlretrieve(url, base_path)
    except Exception:
        pass

base = {"zero_shot_accuracy": 0.0, "zero_shot_tokens": 20.9, "cot_accuracy": 0.0, "cot_tokens": 351.9}
if os.path.exists(base_path):
    with open(base_path, "r", encoding="utf-8") as f:
        base = json.load(f)

print("\n" + "="*65)
print("             OFFICIAL PROJECT COMPARISON TABLE                   ")
print("="*65)
print(f"1. Zero-Shot Direct:        {base['zero_shot_accuracy']:.2f}% (Avg {base['zero_shot_tokens']:.1f} tokens)")
print(f"2. Chain-of-Thought (CoT):  {base['cot_accuracy']:.2f}% (Avg {base['cot_tokens']:.1f} tokens)")
print(f"3. GRPO Fine-Tuned Policy:  {grpo_accuracy:.2f}% (Avg {avg_tokens:.1f} tokens) [{TRAINING_MODE.upper()} mode]")
print("="*65)

# Save final comparison
final_res = {
    "zero_shot_accuracy": base["zero_shot_accuracy"],
    "zero_shot_tokens": base["zero_shot_tokens"],
    "cot_accuracy": base["cot_accuracy"],
    "cot_tokens": base["cot_tokens"],
    "grpo_accuracy": grpo_accuracy,
    "grpo_tokens": avg_tokens,
    "training_mode": TRAINING_MODE
}
with open(os.path.join(results_dir, "final_comparison.json"), "w", encoding="utf-8") as f:
    json.dump(final_res, f, indent=2)
print("Saved final results to results/final_comparison.json")
